## Microbial Source Tracking

using SourceTracker2: https://github.com/caporaso-lab/sourcetracker2/tree/master/data/tiny-test \
and maybe this addon: https://github.com/jordy-code/SourceTracker2-diagnostics/tree/master

Briefly, downloaded available metagenomic data, run through kraken2 and bracken to get OTU table. Then combine with COL otu tables using python script. Once you have a combined OTU table, you should be able to make a separate file the designates the source (sewage) and sink (coral samples) and run sourcetracker2 on these!

In [ ]:
#made a scratch directory for this!
ws_allocate -m nikea_ulrich@uml.edu -r 1 mstracker 30

In [ ]:
#INSTALLATION
module load conda/latest
mkdir envs
conda create --prefix /scratch/workspace/nikea_ulrich_uml_edu-mstracker/envs/st2 -c biocore python=3.8
conda activate /scratch/workspace/nikea_ulrich_uml_edu-mstracker/envs/st2
conda install -c numpy scipy scikit-bio h5py hdf5 seaborn biom-format
pip install git+https://github.com/jordy-code/SourceTracker2-diagnostics.git@Update


In [ ]:
#sourcing wastewater metagenomic data:
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR171/006/ERR1713346/ERR1713346_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/ERR171/006/ERR1713346/ERR1713346_1.fastq.gz

from: https://www.ebi.ac.uk/ena/browser/view/SAMEA4527614

others: https://www.ebi.ac.uk/ena/browser/view/SAMN12001084 (no public data)
https://www.ebi.ac.uk/ena/browser/view/SAMN13927461 (lots of sequence data here!) downloading them to scratch directory

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /scratch/workspace/nikea_ulrich_uml_edu-mstracker/slurm-genome-download-%j.out  # %j = job ID  # %j = job ID

#cd /scratch/workspace/nikea_ulrich_uml_edu-mstracker

wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/028/SRR10990428/SRR10990428_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/029/SRR10990429/SRR10990429_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/026/SRR10990426/SRR10990426_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/032/SRR10990432/SRR10990432_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/033/SRR10990433/SRR10990433_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/030/SRR10990430/SRR10990430_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/026/SRR10990426/SRR10990426_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/034/SRR10990434/SRR10990434_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/036/SRR10990436/SRR10990436_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/032/SRR10990432/SRR10990432_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/031/SRR10990431/SRR10990431_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/034/SRR10990434/SRR10990434_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/030/SRR10990430/SRR10990430_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/033/SRR10990433/SRR10990433_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/028/SRR10990428/SRR10990428_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/027/SRR10990427/SRR10990427_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/037/SRR10990437/SRR10990437_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/036/SRR10990436/SRR10990436_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/027/SRR10990427/SRR10990427_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/037/SRR10990437/SRR10990437_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/035/SRR10990435/SRR10990435_2.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/029/SRR10990429/SRR10990429_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/035/SRR10990435/SRR10990435_1.fastq.gz
wget -nc ftp://ftp.sra.ebi.ac.uk/vol1/fastq/SRR109/031/SRR10990431/SRR10990431_1.fastq.gz

#Job ID: 40034145
#script name: ena-file-download.sh

Total of 14 wastewater/sewage genomes. let's try with these! \
First step is to run these through kraken2!

In [ ]:
#INSTALLATION (in this new scratch directory)
module load conda/latest
conda create --prefix /scratch/workspace/nikea_ulrich_uml_edu-mstracker/envs/kraken2 python=3.8
conda activate /scratch/workspace/nikea_ulrich_uml_edu-mstracker/envs/kraken2
conda install bioconda::kraken2
conda install bracken

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=100G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /scratch/workspace/nikea_ulrich_uml_edu-mstracker/slurm-kraken2-sewage-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /scratch/workspace/nikea_ulrich_uml_edu-mstracker/envs/kraken2

#db already downloaded and built by Brooke
DBNAME="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/ref_databases/standard"
READS="/scratch/workspace/nikea_ulrich_uml_edu-mstracker"
OUTDIR="/scratch/workspace/nikea_ulrich_uml_edu-mstracker/kraken2"
mkdir -p $OUTDIR
LISTPATH="/scratch/workspace/nikea_ulrich_uml_edu-mstracker"
SAMPLELIST="ena_wastewater_sampleids.txt"

THREADS=20
KMER_LEN=35 #(default)
READ_LEN=150

# classify each set of paired end reads against refseq taxonomic database
while IFS= read -r SAMPLEID; do
kraken2 --db $DBNAME --threads $THREADS --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --paired $READS/"${SAMPLEID}"_1.fastq.gz $READS/"${SAMPLEID}"_2.fastq.gz > $OUTDIR/"${SAMPLEID}".kraken2
if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples processed successfully."

conda deactivate

# JOB-ID: 40048083  (40041518: so stupid  didn't distinguish R1s and R2s so needed to redo, script has been edited)
# bash script file name: /bash_scripts/kraken2-sewage.sh

**above script failed after 3 samples...might be a memory thing, but will proceed with these first and if sourcetracker goes well with these few, will re-run this script for the rest of the samples!**

downloading other genomes from SRA, need sra tool-kit \
these might be useful? https://www.ncbi.nlm.nih.gov/sra/SRX5374752[accn] \
Shotgun metagenomics of urban environment: sewage water (8 samples) from Uruguay (can't find any other promising options rn) 

In [ ]:
#INSTALLATION
conda create -n sra-tools
conda activate sra-tools
conda install bioconda::sra-tools

#to download the fastq R1s and R2s with the SRR Accession # (have to do indiv)
fastq-dump --split-files SRR8573788
    #without the split-files, it downloads one fastq that is interleaved
    #prefetch downloads data in a .sra but not don't need this?

#downloaded the other 7 samples: SRR8573789, SRR8573790, SRR8573791, SRR8573793, SRR8573796, SRR8573804, SRR8573807	

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=100G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /scratch/workspace/nikea_ulrich_uml_edu-mstracker/slurm-kraken2-sewage_sra-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /scratch/workspace/nikea_ulrich_uml_edu-mstracker/envs/kraken2

#db already downloaded and built by Brooke
DBNAME="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/ref_databases/standard"
READS="/scratch/workspace/nikea_ulrich_uml_edu-mstracker/sra_genomes"
OUTDIR="/scratch/workspace/nikea_ulrich_uml_edu-mstracker/kraken2_sra"
mkdir -p $OUTDIR
LISTPATH="/scratch/workspace/nikea_ulrich_uml_edu-mstracker"
SAMPLELIST="sra_wastewater_sampleids.txt"

THREADS=20
KMER_LEN=35 #(default)
READ_LEN=150

# classify each set of paired end reads against refseq taxonomic database
while IFS= read -r SAMPLEID; do
kraken2 --db $DBNAME --threads $THREADS --report $OUTDIR/"${SAMPLEID}".kreport2 --report-zero-counts --paired $READS/"${SAMPLEID}"_1.fastq $READS/"${SAMPLEID}"_2.fastq > $OUTDIR/"${SAMPLEID}".kraken2
if [ $? -eq 0 ]; then
        echo "kraken2 completed successfully for sample: $SAMPLEID"
    else
        echo "kraken2 encountered an error for sample: $SAMPLEID"
        exit 1
    fi
done < "$LISTPATH/${SAMPLELIST}"
echo "Kraken2: All samples processed successfully."

conda deactivate

# JOB-ID: 40048877
# bash script file name: /bash_scripts/kraken2-sewage_sra.sh

then ran Bracken on all wastewater kraken2 outputs together

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /scratch/workspace/nikea_ulrich_uml_edu-mstracker/slurm-bracken-wastewater-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /scratch/workspace/nikea_ulrich_uml_edu-mstracker/envs/kraken2

DBNAME="/project/pi_sarah_gignouxwolfsohn_uml_edu/Reference_genomes/ref_databases/standard"
OUTDIR="/scratch/workspace/nikea_ulrich_uml_edu-mstracker/bracken_T1"
mkdir -p $OUTDIR
LISTPATH="/scratch/workspace/nikea_ulrich_uml_edu-mstracker"
SAMPLELIST="all_wastewater_sampleids_T1.txt"

THREADS=20
KMER_LEN=35 #(default)
READ_LEN=150

# Generate bracken database - just do once
#bracken-build -d ${DBNAME} -t ${THREADS} -k ${KMER_LEN} -l ${READ_LEN} 

#${KRAKEN_DB}`  = location of the built Kraken 1/Kraken 2/KrakenUniq database
#${THREADS}`    = number of threads to use with Kraken and the Bracken scripts
#${KMER_LEN}`   = length of kmer used to build the Kraken database 
    #Kraken 1/KrakenUniq default kmer length = 31
    #Kraken 2 default kmer length = 35
    #Default set in the script is 35. 
#${READ_LEN}`   = the read length of your data e.g., if you are using 100 bp reads, set it to `100`. 
#need taxo.k2d, hash.k2d, and opts.k2d. in kraken2 database

# Abundance Estimation with Bracken
KRAKENFILES="/scratch/workspace/nikea_ulrich_uml_edu-mstracker/kraken2_all"
LEVEL="S"

while IFS= read -r line; do
    # Execute bracken command for each file
    bracken -d "$DBNAME" -i $KRAKENFILES/"$line".kreport2 -o $OUTDIR/"${line%.kreport2}_$LEVEL.bracken" -r "$READ_LEN" -l "$LEVEL"
if [ $? -eq 0 ]; then
        echo "bracken completed successfully for sample: $SAMPLEID"
    else
        echo "bracken encountered an error for sample: $SAMPLEID"
        exit 1
    fi
    done < "$LISTPATH/${SAMPLELIST}"
    
conda deactivate

#${SAMPLE}.kreport - the kraken report generated for a given dataset
#{BRACKEN_OUTPUT_FILE}.bracken - the desired name of the output file to be generated by the code
#The following optional parameters may be specified:
    #${LEVEL} - Default = 'S'. This specifies that abundance estimation will calculate estimated reads for each species. Other possible options are K (kingdom level), P (phylum), C (class), O (order), F (family), and G (genus).
    #${THRESHOLD} - Default = 10. For species classification, any species with <= 10 (or otherwise specified) reads will not receive any additional reads from higher taxonomy levels when distributing reads for abundance estimation. 
    #If another classification level is specified, thresholding will occur at that level.

# JOB-ID: 40060831
# bash script file name: /bash_scripts/bracken-wastewater.sh 

ran this before kraken2 was totally done, did on 8 samples (bracken_T1). Just to see get to SourceTracker quicker. Can run again on all samples if it seems to be working!

In [ ]:
#copied all COL samples OTU tables to bracken_T1
cp /work/pi_sarah_gignouxwolfsohn_uml_edu/nikea/COL/kraken_bracken/k_b_outputs/bracken/*.bracken ./

In [ ]:
#merged all OTU tables: wastewater ones with all COL samples!
python combine_bracken_outputs.py --files *.bracken --output COL_wastewater_merged_bracken_species_T1.txt 

**go over to other notebook to format the OTU table and normalize: estimated_abundances_from_kraken_wastewater.ipynb**

using an OTU table that is not normalized for this first pass with SourceTracker2!

need to merge the OTU table and the species ID- did this in R: formating_table_for_MST.R

### Source Tracker

need OTU table.txt and map.txt for inputs. The map file designates which samples are sources and sinks

In [ ]:
cd /scratch/workspace/nikea_ulrich_uml_edu-mstracker/sourcetracker2

In [ ]:
module load conda/latest
conda activate /scratch/workspace/nikea_ulrich_uml_edu-mstracker/envs/st2

In [ ]:
sourcetracker2 gibbs -i otu_normtable_w_species_MST_ready.txt -m map.txt -o output_norm/ 
#had to edit the map.txt such that there weren't any lines after the last line of text (hence map1.txt). 
#Also make sure the column headings are #SampleID in map file and #OTU ID in otu table.
#OTU table should have #Constructed from biom file in the first line. 
#made sure sample names lined up with map.txt file

In [ ]:
sourcetracker2 gibbs -i otu_table_w_species_ready.txt -m map1.txt -o output_diagnostic/ --diagnostics --draws_per_restart 2

It ran! Ok we are obviously limited in what we can detect by the source samples. Try this on all sampleset and also normalize the otu table too

when run on a normalized OTU table, it identifies almost half the sample as sourced from sewage inputs

In [ ]:
# let's try to filter out the identified source samples (see what's left)

sourcetracker2 gibbs -i otu_normtable_w_species_MST_ready_sample_abbr.txt -m map_sample_abbr.txt -o output_for_filtering/ --source_rarefaction_depth 0 --sink_rarefaction_depth 0 --per_sink_feature_assignments
#--per_sink_feature_assignments
#need to include this in a bash script, conked out on the login node

In [ ]:
#!/bin/bash
#SBATCH -c 24  # Number of Cores per Task
#SBATCH --mem=50G  # Requested Memory
#SBATCH -p cpu  # Partition
#SBATCH -t 48:00:00  # Job time limit
#SBATCH --mail-type=ALL
#SBATCH -o /scratch/workspace/nikea_ulrich_uml_edu-mstracker/slurm-sourcetracker-%j.out  # %j = job ID  # %j = job ID

module load conda/latest
conda activate /scratch/workspace/nikea_ulrich_uml_edu-mstracker/envs/st2

OTUTABLE="/scratch/workspace/nikea_ulrich_uml_edu-mstracker/sourcetracker2/otu_normtable_w_species_MST_ready_sample_abbr.txt"
MAP="/scratch/workspace/nikea_ulrich_uml_edu-mstracker/sourcetracker2/map_sample_abbr.txt"
OUTPATH="/scratch/workspace/nikea_ulrich_uml_edu-mstracker/sourcetracker2"

sourcetracker2 gibbs -i $OTUTABLE -m $MAP -o $OUTPATH/output_for_filtering/ --source_rarefaction_depth 0 --sink_rarefaction_depth 0 --per_sink_feature_assignments
  
conda deactivate

# JOB-ID: 40174954
# bash script file name: /bash_scripts/sourcetracker.sh 